# DL_02: Corr2Cause Causal Entailment with Fine-Tuned DistilBERT
**Causal NLP Reasoning — Phase 2: Deep Learning**

---

## 1. Purpose and Scope
This notebook implements fine-tuning of **DistilBERT** (`distilbert-base-uncased`) for binary causal sequence entailment on the **Corr2Cause** benchmark.

Key requirements:
- Frame causal discovery deduction as sequence classification: `[CLS] Premise [SEP] Hypothesis [SEP]`.
- Train on canonical training split (205,734 pairs) with checkpoint selection strictly on the validation split (1,076 pairs).
- Never evaluate or peek at the held-out test split (1,162 pairs) during hyperparameter tuning.
- Optimize using AdamW, linear learning rate warmup, and mixed precision (FP16 only when CUDA is available).
- Use conservative default batch size (16) and gradient accumulation (1).
- Configure modern `eval_strategy='epoch'` (conforming to modern Transformers specifications).
- Compute final evaluation metrics on held-out test split: Accuracy, Macro-F1, Precision, and Recall.
- Export test prediction logits and labels for comparative evaluation in `DL_05`.


## 2. Experimental Methodology & Theoretical Background

### Abstract Causal Discovery as Deductive Entailment
Corr2Cause tests whether language models can determine whether a causal conjecture is mathematically entailed by observed conditional independencies:
- Given variables $V = \{X_1, X_2, \dots, X_n\}$ and statistical statements (e.g. $X_1 \perp X_2 \mid X_3$), determine if $X_i \to X_j$ is uniquely implied.
- The objective is binary sequence classification: Cross-entropy loss over the `[CLS]` pooled classification representation.

### Hardware & Compute Efficiency
- DistilBERT features 66M parameters, trains rapidly, and fits comfortably within free Kaggle (T4 x 2) and Google Colab (T4) memory bounds.
- Canonical split counts: Train = 205,734 | Validation = 1,076 | Test = 1,162.


## 3. Expected Inputs

| Split | Path | Exact Row Count | Columns |
| :--- | :--- | :--- | :--- |
| **Train** | `data/raw/train.csv` (or `data/processed/dl_corr2cause_train.csv`) | 205,734 | `input, label, num_variables, template` |
| **Validation** | `data/raw/validation.csv` (or `data/processed/dl_corr2cause_val.csv`) | 1,076 | `input, label, num_variables, template` |
| **Test** | `data/raw/test.csv` (or `data/processed/dl_corr2cause_test.csv`) | 1,162 | `input, label, num_variables, template` |


## 4. Technical Responsibilities & Implementation Requirements

1. **Portable Root & Hardware Detection**: Check GPU availability (CUDA / MPS / CPU) and locate project root.
2. **Configurable Hyperparameters**: Conservative defaults: Batch size (16), gradient accumulation (1), learning rate ($2 \times 10^{-5}$), weight decay (0.01), epochs (3), FP16 (CUDA only), `eval_strategy='epoch'`.
3. **Tokenization**: Tokenize premise-hypothesis pairs with `distilbert-base-uncased`.
4. **Model Architecture**: Initialize `AutoModelForSequenceClassification` with 2 labels.
5. **Training Execution**: Run training with validation evaluation at each epoch.
6. **Validation Selection**: Select best model weights strictly using validation Macro-F1.
7. **Test Set Evaluation & Export**: Run final test evaluation once and export predictions to `data/processed/`.


In [ ]:
# Section 1: Environment Setup & Portable Imports
import sys
import os
from pathlib import Path

def find_repo_root():
    curr = Path(os.getcwd()).resolve()
    for p in [curr] + list(curr.parents):
        if (p / 'AML' / 'causal_extractors.py').exists() and (p / 'DL' / 'dl_utils.py').exists():
            return p
    return curr

ROOT_DIR = find_repo_root()
for p in [ROOT_DIR, ROOT_DIR / 'AML', ROOT_DIR / 'DL']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import dl_utils
print(f'Repository root located at: {ROOT_DIR}')


In [ ]:
# Section 2: Configurable Hyperparameters & Conservative Resource Defaults
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

MODEL_CHECKPOINT = dl_utils.DISTILBERT_CHECKPOINT
BATCH_SIZE = dl_utils.DEFAULT_DISTILBERT_BATCH_SIZE  # Conservative default: 16
GRAD_ACCUM_STEPS = dl_utils.DEFAULT_DISTILBERT_GRAD_ACCUM  # Default: 1
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01
MAX_LENGTH = 256
FP16_ENABLED = torch.cuda.is_available() and not torch.backends.mps.is_available()
EVAL_STRATEGY = 'epoch'  # Modern Transformers argument
SAVE_TOTAL_LIMIT = dl_utils.DEFAULT_CHECKPOINT_LIMIT
OUTPUT_DIR = str(ROOT_DIR / 'DL' / 'checkpoints' / 'corr2cause_distilbert')


In [ ]:
# Section 3: Dataset Ingestion & Tokenization
# TODO: Load canonical splits (205,734 train, 1,076 val, 1,162 test)
# TODO: Tokenize inputs using AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)


In [ ]:
# Section 4: Model Instantiation
# TODO: Instantiate AutoModelForSequenceClassification with num_labels=2


In [ ]:
# Section 5: Training Pipeline & Validation-Based Selection
# TODO: Configure TrainingArguments with eval_strategy=EVAL_STRATEGY, metric_for_best_model='eval_f1'
# TODO: Execute training loop on free GPU tier


In [ ]:
# Section 6: Held-Out Test Evaluation & Prediction Export
# TODO: Predict on held-out test split (1,162 instances)
# TODO: Export predictions to data/processed/dl_corr2cause_test_predictions.csv
# Required columns: ['index', 'true_label', 'pred_label', 'prob_entailed', 'num_variables']


## 5. Expected Outputs & Evaluation Criteria

- Best model checkpoint (saved locally in temporary checkpoint dir).
- `data/processed/dl_corr2cause_test_predictions.csv`: Model predictions for all 1,162 test instances.
- **Evaluation Criteria**: Test Accuracy, Macro-F1, Precision, and Recall reported overall and disaggregated across variable counts (2, 3, 4 variables).
